In [ ]:
!pip install lightning pytorch-forecasting

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# 1. Check that the shared metric function is available
if "calculate_metrics" not in globals():
    raise RuntimeError(
        "Run Code 7 first to define calculate_metrics()."
    )


# 2. Define the model result directories
model_folders = {
    "GCN": Path("results/step_07"),
    "GAT": Path("results/step_08"),
    "TFT": Path("results/step_09")
}

output_folder = Path(
    "results/step_10"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

expected_targets = pd.read_csv(
    "results/step_05/test_target_identities.csv"
)

identity_columns = [
    "Reference",
    "Origin_Date",
    "Target_Date",
    "Horizon",
    "Actual"
]

sort_columns = [
    "Reference",
    "Horizon"
]

expected_identities = (
    expected_targets[identity_columns]
    .sort_values(sort_columns)
    .reset_index(drop=True)
)


# 3. Load and validate each model's predictions
prediction_tables = []

for model_name, folder in model_folders.items():

    prediction_path = (
        folder / f"{model_name}_test_predictions.csv"
    )

    if not prediction_path.exists():
        raise FileNotFoundError(
            f"Prediction file not found: {prediction_path}"
        )

    model_predictions = pd.read_csv(
        prediction_path
    )

    required_columns = (
        identity_columns
        + ["Model", "Prediction", "Is_Synthetic"]
    )

    missing_columns = (
        set(required_columns)
        - set(model_predictions.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{model_name}: missing columns "
            f"{sorted(missing_columns)}"
        )

    if not model_predictions["Model"].eq(
        model_name
    ).all():
        raise ValueError(
            f"Incorrect model labels in {prediction_path}"
        )

    if model_predictions.duplicated(
        ["Reference", "Origin_Date", "Horizon"]
    ).any():
        raise ValueError(
            f"{model_name}: duplicate forecast identities."
        )

    numeric_values = model_predictions[
        ["Actual", "Prediction"]
    ].to_numpy(dtype=float)

    if not np.isfinite(numeric_values).all():
        raise ValueError(
            f"{model_name}: non-finite actual or predicted values."
        )

    if (numeric_values < 0).any():
        raise ValueError(
            f"{model_name}: negative quantities were found."
        )

    if not (
        model_predictions["Is_Synthetic"]
        .astype(str)
        .str.lower()
        .eq("true")
        .all()
    ):
        raise ValueError(
            f"{model_name}: synthetic data labels are inconsistent."
        )

    actual_identities = (
        model_predictions[identity_columns]
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        actual_identities,
        expected_identities,
        check_dtype=False,
        check_exact=False,
        rtol=1e-6,
        atol=1e-6
    )

    prediction_tables.append(
        model_predictions
    )

    print(
        f"{model_name}: verified "
        f"{len(model_predictions)} common test targets."
    )


# 4. Combine the predictions
all_predictions = pd.concat(
    prediction_tables,
    ignore_index=True
)

all_predictions.to_csv(
    output_folder / "all_model_predictions.csv",
    index=False
)


# 5. Recalculate overall metrics from the prediction files
comparison_records = []

for model_name, group in all_predictions.groupby("Model"):

    recalculated_metrics = calculate_metrics(
        group["Actual"],
        group["Prediction"]
    )

    comparison_records.append({
        "Model": model_name,
        **recalculated_metrics
    })

    # Verify agreement with the metrics saved during model evaluation.
    saved_metrics = pd.read_csv(
        model_folders[model_name]
        / f"{model_name}_test_metrics.csv"
    )

    if len(saved_metrics) != 1:
        raise ValueError(
            f"{model_name}: expected one overall metrics row."
        )

    for metric_name, recalculated_value in (
        recalculated_metrics.items()
    ):

        saved_value = saved_metrics.iloc[0][
            metric_name
        ]

        if recalculated_value is None:

            if not pd.isna(saved_value):
                raise ValueError(
                    f"{model_name}: inconsistent {metric_name}."
                )

        else:

            np.testing.assert_allclose(
                float(saved_value),
                float(recalculated_value),
                rtol=1e-6,
                atol=1e-6,
                err_msg=(
                    f"{model_name}: metric mismatch "
                    f"for {metric_name}"
                )
            )

comparison_df = pd.DataFrame(
    comparison_records
)

model_order = [
    "GCN",
    "GAT",
    "TFT"
]

comparison_df = (
    comparison_df
    .set_index("Model")
    .loc[model_order]
    .reset_index()
)

comparison_df.to_csv(
    output_folder / "model_comparison_metrics.csv",
    index=False
)

print("\nOverall test metrics:")
print(
    comparison_df.round(4).to_string(index=False)
)


# 6. Calculate metrics separately for each forecast horizon
horizon_records = []

for (model_name, horizon), group in (
    all_predictions.groupby(
        ["Model", "Horizon"]
    )
):

    horizon_records.append({
        "Model": model_name,
        "Horizon": int(horizon),
        **calculate_metrics(
            group["Actual"],
            group["Prediction"]
        )
    })

horizon_comparison_df = pd.DataFrame(
    horizon_records
)

horizon_comparison_df.to_csv(
    output_folder / "model_comparison_by_horizon.csv",
    index=False
)


# 7. Report rankings separately for each metric
metric_names = [
    "MAE",
    "RMSE",
    "MAPE_nonzero_pct",
    "WAPE_pct"
]

ranking_records = []

for metric in metric_names:

    ordered_models = comparison_df[
        ["Model", metric]
    ].dropna().sort_values(metric)

    ranks = ordered_models[metric].rank(
        ascending=True,
        method="min"
    )

    for (_, row), rank in zip(
        ordered_models.iterrows(),
        ranks
    ):

        ranking_records.append({
            "Metric": metric,
            "Model": row["Model"],
            "Value": float(row[metric]),
            "Rank": int(rank)
        })

ranking_df = pd.DataFrame(
    ranking_records
)

ranking_df.to_csv(
    output_folder / "model_rankings_by_metric.csv",
    index=False
)

print("\nModel rankings by metric:")
print(
    ranking_df.round(4).to_string(index=False)
)


# 8. Plot the overall metric comparison
colors = {
    "GCN": "#4472C4",
    "GAT": "#ED7D31",
    "TFT": "#70AD47"
}

metric_labels = {
    "MAE": "MAE — quantity units",
    "RMSE": "RMSE — quantity units",
    "MAPE_nonzero_pct": "MAPE — nonzero targets (%)",
    "WAPE_pct": "WAPE (%)"
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 8)
)

for ax, metric in zip(
    axes.flat,
    metric_names
):

    ax.bar(
        comparison_df["Model"],
        comparison_df[metric],
        color=[
            colors[name]
            for name in comparison_df["Model"]
        ]
    )

    ax.set_title(
        metric_labels[metric]
    )

    ax.set_ylabel(
        metric_labels[metric]
    )

    ax.grid(
        axis="y",
        alpha=0.25
    )

fig.suptitle(
    "GCN, GAT, and TFT comparison "
)

fig.tight_layout()

fig.savefig(
    output_folder / "overall_model_comparison.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 9. Plot prediction errors across the six horizons
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

for ax, metric in zip(
    axes,
    ["MAE", "RMSE"]
):

    for model_name in model_order:

        model_results = (
            horizon_comparison_df.loc[
                horizon_comparison_df["Model"]
                == model_name
            ]
            .sort_values("Horizon")
        )

        ax.plot(
            model_results["Horizon"],
            model_results[metric],
            marker="o",
            color=colors[model_name],
            label=model_name
        )

    ax.set_xlabel("Forecast horizon — months")
    ax.set_ylabel(metric)
    ax.set_xticks(range(1, 7))
    ax.set_title(f"{metric} by forecast horizon")
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    "Forecast horizon comparison — data"
)

fig.tight_layout()

fig.savefig(
    output_folder / "errors_by_forecast_horizon.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 10. Plot the total forecast across all parts
# Actual demand is counted once, not once per model.
actual_totals = (
    all_predictions.loc[
        all_predictions["Model"] == model_order[0]
    ]
    .groupby("Target_Date")["Actual"]
    .sum()
    .sort_index()
)

fig, ax = plt.subplots(
    figsize=(10, 5)
)

ax.plot(
    pd.to_datetime(actual_totals.index),
    actual_totals.values,
    color="black",
    linewidth=2,
    marker="o",
    label="Actual demand"
)

for model_name in model_order:

    forecast_totals = (
        all_predictions.loc[
            all_predictions["Model"] == model_name
        ]
        .groupby("Target_Date")["Prediction"]
        .sum()
        .sort_index()
    )

    ax.plot(
        pd.to_datetime(forecast_totals.index),
        forecast_totals.values,
        marker="o",
        color=colors[model_name],
        label=model_name
    )

ax.set_xlabel("Target month")
ax.set_ylabel("Total quantity across all parts")
ax.set_title(
    "Aggregate six-month forecasts — data"
)

ax.legend()
ax.grid(alpha=0.25)

fig.autofmt_xdate()
fig.tight_layout()

fig.savefig(
    output_folder / "aggregate_forecast_comparison.png",
    dpi=160
)

plt.show()
plt.close(fig)


# 11. Save the comparison protocol
protocol = {
    "models": model_order,
    "common_test_targets_per_model": len(
        expected_targets
    ),
    "forecast_horizons": sorted(
        expected_targets["Horizon"]
        .astype(int)
        .unique()
        .tolist()
    ),
    "metrics_recomputed_from_predictions": True,
    "saved_metrics_consistency_verified": True,
    "target_identities_verified": True,
    "prediction_units": "Original quantity units",
    "mape_policy": (
        "Exclude zero actual targets only from MAPE. "
        "Retain them in MAE, RMSE, MSE, and WAPE."
    ),
    "interpretation": (
        "Lower errors indicate better performance on this "
        "test set. Rankings may differ by metric. "
        "No statistical significance claim is made."
    )
}

(
    output_folder / "comparison_protocol.json"
).write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print(
    "\nPassed: all models use the same test targets, "
    "and saved metrics match the recalculated values."
)

print(f"\nResults saved in: {output_folder}")